In [ ]:
%pip install psycopg2
%pip install pandas
%pip install requests

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import psycopg2
import psycopg2.extras
import pandas as pd
import requests
import json
import time
from datetime import datetime
import csv
import os.path

In [31]:
# use for load station data


# api_url = "http://localhost/thaiwater/rain_yearly.json"   # old
api_url = "https://api-v3.thaiwater.net/api/v1/thaiwater30/public/rain_yearly"   # new from website
station_file = "thaiwater_station.csv"



In [ ]:
#for load station data and save to csv file
response = requests.get(api_url)
print("response status: %d" % response.status_code)
if (response.status_code == 200):

    response_json =response.json()
    data_list = response_json["data"]
    data_size = len(data_list)
    print("insert: %d" % data_size)
    # [note 2] บางสถานีมีตัวอักษรที่ tis-620 เขียนไม่ได้ (ภาษาลาว,   non-breaking space) -> UnicodeEncodeError
    #           เพิ่ม errors="replace" ให้ตัวที่เขียนไม่ได้กลายเป็น '?' แทนการ error
    # with open(station_file, 'w', encoding="tis-620", newline='') as file:   # old
    with open(station_file, 'w', encoding="tis-620", errors="replace", newline='') as file:   # new
        writer = csv.writer(file)
        header_row = ["station_id","station_name","lat","lon","agency_name","province","amphoe","tumbon"]
        writer.writerow(header_row)
        for dd in data_list:
            print (dd['station']['id'],dd['station']['tele_station_name']['th'])
            drow = [dd['station']['id'],dd['station']['tele_station_name']['th'],dd['station']['tele_station_lat'],dd['station']['tele_station_long'],dd['agency']['agency_name']['th'],dd['geocode']['province_name']['th'],dd['geocode']['amphoe_name']['th'],dd['geocode']['tumbon_name']['th']]
            
            writer.writerow(drow)

        

In [33]:
#load daily data for each station, loop in month

daily_api_url = "https://api-v3.thaiwater.net/api/v1/thaiwater30/public/rain_monthly_graph?station_id=#id&month=#month&year=#year"
year = 2026
daily_header = ["station_id","station_name","lat","lon","agency_name","province","amphoe","tumbon","date","rain(mm)"]

os.makedirs("thaiwater_daily", exist_ok=True)   # make directory if not exist   

# [note 11] เดิมเปิดไฟล์แบบ 'a' (append) -> รันซ้ำแล้วแถวซ้ำ ต้องลบโฟลเดอร์ก่อนรันใหม่
#   ใหม่: อัปเดตแบบเขียนทับ โดยเช็กก่อนว่าในไฟล์ของวันนั้น (1 ไฟล์ = 1 วัน) มีสถานีนี้แล้วหรือยัง
#         มีแล้ว -> ทับแถวเดิมด้วยค่าใหม่ | ยังไม่มี -> เพิ่มแถวใหม่ | สถานีอื่นที่มีอยู่ในไฟล์ -> เก็บไว้เหมือนเดิม
#         แล้วเขียนทั้งไฟล์ใหม่ ('w') -> รันซ้ำกี่รอบก็ไม่มีแถวซ้ำ
#   เปลี่ยนให้วนเดือนเป็นวงนอก: ดึงทุกสถานีของเดือนนั้นเก็บไว้ใน memory ก่อน แล้วค่อยเขียนไฟล์ทีละวัน
#   (ถ้าเขียนทุกครั้งที่ได้ข้อมูล 1 สถานี ต้องอ่าน/เขียนไฟล์ทั้งวัน ~4,000 x 365 รอบ ช้ามาก)

def read_daily_file(data_file):
    rows = {}   # station_id -> row
    if (os.path.isfile(data_file)):
        with open(data_file, encoding="tis-620", newline='') as file:
            csv_reader = csv.reader(file)
            next(csv_reader, None)  # skip the headers
            for r in csv_reader:
                rows[r[0]] = r   # ไฟล์เก่าที่มีแถวซ้ำ (จาก append เดิม) จะเหลือแถวเดียว
    return rows

# with open(station_file) as csv_file:   # old
with open(station_file, encoding="tis-620") as csv_file:   # new
    csv_reader = csv.reader(csv_file, delimiter=',')
    next(csv_reader, None)  # skip the headers
    stations = list(csv_reader)

for mm in range(1,13):
    print("month:", mm)
    month_data = {}   # date -> {station_id: row}
    for ii, row in enumerate(stations, 1):
        station_id = row[0]
        if (ii % 200 == 0):
            print("  ", ii, "/", len(stations), "Station:",station_id,row[1])
        api = daily_api_url.replace("#id",station_id).replace("#month",str(mm)).replace("#year",str(year))
        response = requests.get(api)
        if (response.status_code == 200):
            response_json =response.json()
            data_list = response_json["data"]
            for dd in data_list:
                drow = [station_id,row[1],row[2],row[3],row[4],row[5],row[6],row[7],dd["rainfall_datetime"],dd["rainfall_value"]]
                month_data.setdefault(dd["rainfall_datetime"], {})[station_id] = drow

    for data_date, new_rows in sorted(month_data.items()):
        data_file = "thaiwater_daily/"+data_date+".csv"
        rows = read_daily_file(data_file)   # เช็กวันที่: มีไฟล์ของวันนี้อยู่แล้ว -> อ่านแถวเดิมมาก่อน
        rows.update(new_rows)               # เช็กสถานี: มีแล้ว = ทับ, ยังไม่มี = เพิ่ม
        # with open(data_file, 'a', encoding="tis-620", newline='') as file:   # old
        with open(data_file, 'w', encoding="tis-620", errors="replace", newline='') as file:   # new
            writer = csv.writer(file)
            writer.writerow(daily_header)
            writer.writerows(rows.values())
    print("month", mm, "saved", len(month_data), "daily files")

1 Station: 75 ทต.บ้านกล้วย
2 Station: 91 ทต.แพรกศรีราชา
3 Station: 132 เขาคิชฌกูฏ
4 Station: 198 อบต.หันทราย
5 Station: 289 อบต.ทุ่งนางโอก
6 Station: 290 ป่าติ้ว
7 Station: 292 อบต.บึงแก
8 Station: 324 คำปลาหลาย
9 Station: 331 ท้ายฝายชนบท
10 Station: 332 เหนือฝายชนบท
11 Station: 381 อบต.โคกเครือ
12 Station: 382 อบต.เนินยาง
13 Station: 419 ทต.จอมแจ้ง
14 Station: 423 อบต.ท่าตอน
15 Station: 528788 ปากคลองตลาด
16 Station: 528833 จุดวัด ค.แสนแสบ-วัดทรัพย์ฯ
17 Station: 487 ทต.ป่าแมต
18 Station: 509 อบต.ป่าคา
19 Station: 514 อบต.อ่ายนาไลย
20 Station: 551 ทต.แม่สรวย
21 Station: 586 ศาลเจ้าไก่ต่อ (TSK.2)
22 Station: 598 สว่างอารมณ์
23 Station: 615 ทต.ขาณุวรลักษณ์บุรี
24 Station: 646 ทต.กงไกรลาศ
25 Station: 648 สถานีดับเพลิง ทต.หาดเสี้ยว
26 Station: 686 ทต.สามง่าม
27 Station: 764 อบต.ไร่ใหม่พัฒนา
28 Station: 766 อบต.กลัดหลวง
29 Station: 3606 สะพานน้ำแม่สิน
30 Station: 3630 สะพานน้ำงิม
31 Station: 3628 หน่วยศึกษาการพัฒนาการอนุรักษ์ต้นน้ำแม่สะลา
32 Station: 3618 หน่วยศึกษาการพัฒนาการอนุรักษ์ต้นน้ำ

KeyboardInterrupt: 

In [32]:
#load currently --> today data for each station yeah
# [note 5] new cell: ดึงฝน "ของวันนี้" ของทุกสถานี (ทั้ง cell นี้เป็นโค้ดใหม่)
#   - ใช้ API rain_24h: เรียกครั้งเดียวได้ทุกสถานี (~4,300 แห่ง) ใช้เวลาไม่กี่วินาที
#   - rain_24h = ฝนสะสม 24 ชั่วโมงล่าสุด นับถอยหลังจากเวลาที่สถานีส่งข้อมูลล่าสุด (คอลัมน์ datetime)
#   - วิธีใช้: รัน cell 1 (import) แล้วรัน cell นี้ได้เลย ไม่ต้องรัน cell 3, 4
#   - รันซ้ำเมื่อไหร่ก็ได้ จะได้ไฟล์ใหม่ตามเวลาที่รัน (ไม่ทับ ไม่ซ้ำ)
today_api_url = "https://api-v3.thaiwater.net/api/v1/thaiwater30/public/rain_24h"
os.makedirs("thaiwater_today", exist_ok=True)
now_str = datetime.now().strftime("%Y-%m-%d_%H%M")
today_file = "thaiwater_today/rain_24h_" + now_str + ".csv"

response = requests.get(today_api_url, timeout=60)
print("response status: %d" % response.status_code)
if (response.status_code == 200):
    data_list = response.json()["data"]
    print("stations: %d" % len(data_list))
    with open(today_file, 'w', encoding="tis-620", errors="replace", newline='') as file:
        writer = csv.writer(file)
        header_row = ["station_id","station_name","lat","lon","agency_name","province","amphoe","tumbon","datetime","rain_24h(mm)","rain_1h(mm)"]
        writer.writerow(header_row)
        for dd in data_list:
            drow = [dd['station']['id'],dd['station']['tele_station_name']['th'],dd['station']['tele_station_lat'],dd['station']['tele_station_long'],dd['agency']['agency_name']['th'],dd['geocode']['province_name']['th'],dd['geocode']['amphoe_name']['th'],dd['geocode']['tumbon_name']['th'],dd['rainfall_datetime'],dd.get('rain_24h'),dd.get('rain_1h')]
            writer.writerow(drow)
    print("saved:", today_file)

response status: 200
stations: 4365
saved: thaiwater_today/rain_24h_2026-09-30_0346.csv
